In [1]:
!pip install -q fastparquet
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 36.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 2.7 MB/s eta 0:00:00


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA, HoltWinters

In [3]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [4]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'erratic' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand

,unique_id,ds,y
22,141905824,2024-01-01,1.0
23,142629362,2024-01-01,1.0
30,142954403,2024-01-01,1.0
33,147282213,2024-01-01,1.0
92,215482427,2024-01-01,1.0
...,...,...,...
5266140,1656117213,2025-09-30,70.0
5266153,1656116903,2025-09-30,86.0
5266155,160506881,2025-09-30,89.0
5266158,1933002627,2025-09-30,95.0


In [27]:
# 1. Создаем список с моделями
models = [
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),
    Naive(alias='Naive'),
    AutoETS(alias='AutoETS'),
    AutoARIMA(alias='AutoARIMA'),
    OptimizedTheta(alias='OptTheta'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA')
]

# TSB - эталон для lumpy рядов
# ETS - семейство экспоненциальных сглаживаний
# SeasonalNaive - примитивный лаговый прогноз
# Naive - прогноз это последнее значение ряда
# Theta - разбивает ряд на компоненты (линии ряда), каждая компонента прогнозируется отдельно, потом прогнозы комбинируются для получения результата
# ARIMA - семейство ARIMA моделей

In [28]:
# задаем параметры
horizon = 14
step_size = 7
n_windows = 5

# определяем минимально необходимую длину ряда
min_required = n_windows * step_size + horizon

# Группировка и подсчёт длины
series_len = real_demand.groupby('unique_id').size()
long_series = series_len[series_len >= min_required].index

# Оставляем только длинные ряды
real_demand_filtered = real_demand[real_demand['unique_id'].isin(long_series)]

# Создаём экземпляр StatsForecast со списком моделей
sf = StatsForecast(models=models, freq='D', n_jobs=1, verbose = True, fallback_model=Naive())

# Теперь запускаем кросс-валидацию на отфильтрованных данных
cv_results = sf.cross_validation(
    df=real_demand_filtered,
    h=horizon,
    step_size=step_size,
    n_windows=n_windows
)

Cross Validation Time Series 1:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 2:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 3:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 4:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 5:   0%|          | 0/5 [00:00<?, ?it/s]

...

Cross Validation Time Series 758:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 759:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 760:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 761:   0%|          | 0/5 [00:00<?, ?it/s]

Cross Validation Time Series 762:   0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
cv_results

,unique_id,ds,cutoff,y,SeasonalNaive7,Naive,AutoETS,AutoARIMA,OptTheta,ADIDA,IMAPA
0,140322963,2025-08-20,2025-08-19,6.0,4.0,3.0,2.610655,2.975122,2.449317,2.568764,2.49151
1,140322963,2025-08-21,2025-08-19,1.0,3.0,3.0,2.610655,2.813935,2.452822,2.568764,2.49151
2,140322963,2025-08-22,2025-08-19,1.0,4.0,3.0,2.610655,2.830689,2.456326,2.568764,2.49151
3,140322963,2025-08-23,2025-08-19,3.0,0.0,3.0,2.610655,2.835361,2.459831,2.568764,2.49151
4,140322963,2025-08-24,2025-08-19,3.0,0.0,3.0,2.610655,2.840033,2.463335,2.568764,2.49151
...,...,...,...,...,...,...,...,...,...,...,...
53335,2640085912,2025-09-26,2025-09-16,2.0,1.0,1.0,1.490625,2.748398,2.179360,1.830080,1.83008
53336,2640085912,2025-09-27,2025-09-16,3.0,3.0,1.0,1.490625,2.748451,2.238215,1.830080,1.83008
53337,2640085912,2025-09-28,2025-09-16,7.0,1.0,1.0,1.490625,2.748470,2.297070,1.830080,1.83008
53338,2640085912,2025-09-29,2025-09-16,7.0,1.0,1.0,1.490625,2.748478,2.355925,1.830080,1.83008


In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------------------------------
# 1. Определение метрик
# -------------------------------

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true) + np.abs(y_pred)
    mask = denominator > 0
    if mask.sum() == 0:
        return np.nan
    return (2 * np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]).mean() * 100

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true).sum()
    if denominator == 0:
        return np.nan
    return np.abs(y_true - y_pred).sum() / denominator * 100

# -------------------------------
# 2. Функция для вычисления метрик по каждому окну (без unique_id)
# -------------------------------

def compute_metrics_per_window(cv_results, model_columns, metrics_dict):
    """
    cv_results: DataFrame от sf.cross_validation()
    model_columns: список имён столбцов с прогнозами моделей
    metrics_dict: словарь вида {'mae': mae, 'rmse': rmse, ...}
    Возвращает DataFrame с колонками: cutoff, model, и каждой метрикой.
    Для каждого cutoff (окна) метрика вычисляется по всем точкам всех unique_id.
    """
    records = []
    grouped = cv_results.groupby('cutoff')   # группируем только по окну
    
    for cutoff, group in grouped:
        y_true = group['y'].values
        for model in model_columns:
            y_pred = group[model].values
            row = {'cutoff': cutoff, 'model': model}
            for metric_name, metric_func in metrics_dict.items():
                row[metric_name] = metric_func(y_true, y_pred)
            records.append(row)
    
    return pd.DataFrame(records)

# -------------------------------
# 3. Применение к вашим данным
# -------------------------------

# Предполагаем, что cv_results уже получен после cross_validation
# Определяем столбцы моделей (все, кроме служебных)
model_columns = [col for col in cv_results.columns 
                 if col not in ['unique_id', 'ds', 'cutoff', 'y']]

metrics = {
    'mae': mae,
    'rmse': rmse,
    'smape': smape,
    'wape': wape
}

# Вычисляем метрики для каждого окна (без разбивки по unique_id)
metrics_per_window = compute_metrics_per_window(cv_results, model_columns, metrics)

# -------------------------------
# 4. Агрегация результатов
# -------------------------------

# 4.1. Среднее по всем окнам для каждой модели
summary_mean = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].mean()


# 4.2. Детальная статистика (mean, std, min, max) по окнам
summary_stats = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].agg(['mean', 'std', 'min', 'max'])


summary_mean

,mae,rmse,smape,wape
model,,,,
ADIDA,3.836254,5.589711,75.630421,81.413007
AutoARIMA,4.061213,5.951473,77.337605,84.669247
AutoETS,4.089220,6.166802,77.164126,83.082808
IMAPA,3.846970,5.601466,75.710643,81.604456
Naive,4.594491,6.502619,92.319522,99.474132
OptTheta,4.029831,5.861351,78.553647,84.279532
SeasonalNaive7,4.685972,6.988374,93.739376,95.424916
